In [ ]:
import numpy as np
import pandas as pd

In [ ]:
anime = pd.read_csv('anime.csv')
rating = pd.read_csv('rating.csv')

In [ ]:
anime.head(50)

In [ ]:
anime['genre']

In [ ]:
rating.head(1)

In [ ]:
anime.shape

In [ ]:
rating.shape

In [ ]:
anime.info()
rating.info()

anime.isnull().sum()
rating.isnull().sum()

In [ ]:
rating = rating[rating['rating'] != -1]

In [ ]:
df = rating.merge(anime, on='anime_id')

In [ ]:
df.head(1)

In [ ]:
df.shape

In [ ]:
df = df.rename(columns={
    'rating_x':'user_rating',
    'rating_y':'anime_rating'
})

In [ ]:
df.drop(['type', 'episodes'], axis=1, inplace=True)

In [ ]:
df= df.drop(columns=['members'])

In [ ]:
df.head(1)

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.duplicated().sum()

In [ ]:
df['user_id'].nunique()

In [ ]:
df['anime_id'].nunique()

In [ ]:
df['genre'].isnull().sum()

In [ ]:
df['genre']=df['genre'].apply(lambda x:x.split(', '))

In [ ]:
def collapse(L):
    return [i.replace(" ","") for i in L]
df['genre'] = df['genre'].apply(collapse)

In [ ]:
df['tags'] = df['genre'].apply(lambda x: " ".join(x))

In [ ]:
df['tags'] = df['tags'].str.replace('Sci-Fi', 'SciFi')

In [ ]:
df['tags'].head()

In [ ]:
df['tags'] = df['tags'].apply(lambda x: x.lower())

In [ ]:
##CONTENT BASED RECOMMENDATION ON GENRE

In [ ]:
anime_df = df[['anime_id','name','genre']].drop_duplicates()

In [ ]:
anime_df.shape

In [ ]:
anime_df.dropna(inplace=True)

anime_df['genre'] = anime_df['genre'].apply(lambda x: x.split(', '))

def collapse(L):
    return [i.replace(" ","") for i in L]

anime_df['genre'] = anime_df['genre'].apply(collapse)

anime_df['tags'] = anime_df['genre'].apply(lambda x: " ".join(x))

anime_df['tags'] = anime_df['tags'].str.replace('-', '', regex=False)

anime_df['tags'] = anime_df['tags'].str.lower()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(anime_df['tags']).toarray()

In [ ]:
cv.get_feature_names_out()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity = cosine_similarity(vectors)

In [ ]:
similarity[0]

In [ ]:
def recommend_genre(anime):
    anime = anime.strip()
    anime_index = anime_df[anime_df['name']==anime].index[0]
    distances=similarity[anime_index]
    anime_list = sorted(list(enumerate(distances)),key=lambda x:x[1],reverse=True)[1:6]
    recommendations=[]
    for i in anime_list:
        recommendations.append(
            anime_df.iloc[i[0]]['name']
        )

    return recommendations
    ##print(anime_df.iloc[i[0]]['name'])

In [ ]:
recommend_genre("Death Note")

In [ ]:
##COLLABERATIVE FILTERING

In [ ]:
df

In [ ]:
rating_counts = df.groupby('name')['user_rating'].count()

In [ ]:
popular_anime=rating_counts[rating_counts>=50].index
filter_df=df[df['name'].isin(popular_anime)]

In [ ]:
pt = filter_df.pivot_table(index='name',columns='user_id',values='user_rating')

In [ ]:
pt.fillna(0,inplace=True)

In [ ]:
from scipy.sparse import csr_matrix
anime_sparse = csr_matrix(pt)

In [ ]:
from sklearn.neighbors import NearestNeighbors

In [ ]:
model = NearestNeighbors(metric='cosine', algorithm= 'brute')

In [ ]:
model.fit(anime_sparse)

In [ ]:
import numpy as np
def recommend_collab(anime_name):
    anime_index = np.where(pt.index==anime_name)[0][0]
    distances,suggestions = model.kneighbors(pt.iloc[anime_index,:].values.reshape(1,-1),n_neighbors=6)
    recommendation=[]
    for i in suggestions[0][1:]:
        recommendation.append(
            pt.index[i]
        )
    return recommendation

In [ ]:
recommend_collab("Death Note")

In [ ]:
def recommend_hybrid(anime):
    content_results = recommend_genre(anime)
    collab_results = recommend_collab(anime)
    hybrid = content_results + collab_results
    hybrid = list(dict.fromkeys(hybrid))
    return hybrid[:10]

In [ ]:
recommend_hybrid("Death Note")